In [1]:
import numpy as np
from qutip import *

H = Qobj(np.array([
    [0.70710678+0.j,  0.70710678+0.j],
    [0.70710678+0.j, -0.70710678+0.j]
    ]))

zero = basis(2,0)
one = basis(2,1)
plus = H * zero
minus = H * one

states = [zero,one,plus,minus]



def toy_decoherence_event(reference_state):
    if reference_state == zero or reference_state == one:
        return reference_state
    elif reference_state == plus:
        return  Qobj(np.array([
                [0.5, 0.4999],
                [0.4999, 0.5]
                ]))
    else:
        return  Qobj(np.array([
                [0.5, -0.4999],
                [-0.4999, 0.5]
                ]))

def uPauli_for_noise_sampling(Pauli,U,U0,state1,state2):
    ################################################
    # This method is from an unpublished work of someone else
    ################################################
    temp = state1*state2.dag()
    temp = U0.inv() * temp * U0.inv().dag()
    mixed = U(temp)

    temp1 = Pauli * mixed * Pauli.dag()
    temp1 = state1.dag() * temp1  * state2

    return temp1[0,0]


d_mixed = {'X':[],'Y':[],"Z":[]}
for state1_name, state1 in zip(['0','1','+','-'],states):
    for state2_name, state2 in zip(['0','1','+','-'],states):
        for error_name,Pauli in zip(['X','Y','Z'],[sigmax(),sigmay(),sigmaz()]):
            uP_mixed = uPauli_for_noise_sampling(Pauli = Pauli,
                U = toy_decoherence_event,
                U0 = qeye(2),
                state1 = state1,
                state2 = state2)
            d_mixed[error_name].append(uP_mixed)


for key, val in d_mixed.items():
    print(key+' '+str(np.square(abs(np.mean(val)))))


X 0.003910023093147335
Y 0.13267552523459397
Z 0.13267552523459397
